# 06 — One pixel's optical season, raw and smoothed

Give an **AOI number** and a **pixel id** (`pid`, read from `<aoi>_pixel_index.tif` in QGIS with
*Identify*, or from any output of `optical_phenology`). This notebook draws that pixel's season
exactly as the detector sees it:

- **grey points** — the raw Sentinel-2 observations that passed the clear-score test. The gaps
  between them are cloud.
- **coloured line** — the smoothed curve the detector actually reads: the same 5-day grid and the
  same Whittaker fit (lambda 2, second differences). Where two clear observations are more than
  `MAX_GAP_DAYS` apart the line **breaks**, because nothing was observed there and nothing is
  invented.
- **dashed lines** — the detected `start`, `emergence`, `peak` and `harvest` of the pixel's largest
  cycle. `emergence` is where the rise accelerates hardest; `harvest` is halfway down the fall.
- **LSWI panel** — only when the AOI was exported with the SWIR band. LSWI is the one index here
  that says anything about water; NDWI does not, over these fields it is a near-mirror of NDVI.

**If the dashed lines disagree with the points, the detector is wrong for that pixel.** That is what
this notebook is for.

The first call for an AOI reads its images and takes a few seconds; after that, changing `PID` and
re-running is instant, so you can flick through pixels.


## 1. Choose the pixel

In [ ]:
AOI = 116              # AOI number
PID = 115698             # pixel id from <aoi>_pixel_index.tif
BACKEND = "plotly"     # "plotly" (hover for exact dates and values) or "matplotlib" (static)

# Leave these alone unless you are testing how sensitive the result is to them.
CLEAR_MIN    = 75      # Cloud Score+ x 100 a date must reach to be used
MAX_GAP_DAYS = 35      # wider gaps between clear dates are left empty, not interpolated
LMBD         = 2.0     # Whittaker smoothing strength; higher is smoother
FOLDER       = None    # None picks the SWIR export when this AOI has one, else the plain one

# Figure size. None uses the defaults.
#   matplotlib: inches  plotly: pixels
FIG_WIDTH = None
FIG_HEIGHT = None

## 2. Draw it

In [ ]:
import importlib
import sys
from pathlib import Path

# Find the repository without changing the working directory: the library resolves its own
# repo-relative paths. Keep this repo's src first on sys.path, because another project's
# `sar_pipeline` may be installed in the same kernel — pick the "sar-rice-mapper (.venv)" kernel
# if the check below prints False.
root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

import sar_pipeline
from sar_pipeline.analysis import optical_phenology, pixel_curve as pc

# Reload BOTH, in dependency order. Reloading pixel_curve alone leaves the old optical_phenology
# behind it, and the figure then shows stale settings (an old smoothing strength, an old rule)
# while the title claims otherwise — which is exactly how a fixed bug appears to still be there.
importlib.reload(optical_phenology)
importlib.reload(pc)
pc.forget()   # drop cached image cubes so new settings actually take effect

print("this repository's package:",
      Path(sar_pipeline.__file__).resolve().parent == (root / "src" / "sar_pipeline").resolve())
print("smoothing default λ =", optical_phenology.LMBD)

In [ ]:
fig = pc.show(
    AOI, PID,
    backend=BACKEND,
    width=FIG_WIDTH, height=FIG_HEIGHT,
    clear_min=CLEAR_MIN, max_gap_days=MAX_GAP_DAYS, lmbd=LMBD, folder=FOLDER,
)
fig  # matplotlib shows automatically; plotly renders here

In [ ]:
from sar_pipeline.analysis import pixel_curve as pc

In [ ]:
pc.show(146, 8081, with_sar=True, panels=("VH", "VV", "VHmVV"))

## 3. The numbers behind the lines

One row per cycle found in this pixel. The dashed lines above belong to the row with the largest
`amplitude`. A cycle marked `complete = False` ran into the start or the end of the observed period,
so its length is a floor, not a measurement.

In [ ]:
pc.cycles(AOI, PID, clear_min=CLEAR_MIN, max_gap_days=MAX_GAP_DAYS, lmbd=LMBD, folder=FOLDER).T

## 4. Flicking through pixels

Change `PID` in step 1 and re-run step 2 — the images are already in memory, so it is instant.

To compare a few at once:

```python
for pid in (4043, 8081, 13738, 642):
    pc.show(AOI, pid, backend="matplotlib")
```

After re-exporting imagery for an AOI, call `pc.forget()` once so the cached cube is re-read.

The raw and smoothed values themselves, if you want to work with them rather than look at them:

```python
d = pc.series(AOI, PID)
d["raw"]      # clear observations only: date, ndvi, ndwi, lswi
d["smooth"]   # the 5-day grid the detector reads
d["main"]     # the largest cycle, as a dict
```
